# Setup Environment

In [1]:
# #@title Install Dependencies
# %%capture
# pip install pytorch-lightning torchmetrics

In [2]:
#@title Import Dependencies

import os

import random


from datetime import datetime

from matplotlib import pyplot as plt

import numpy as np

from tqdm import tqdm
# from skimage import io
from PIL import Image
import matplotlib as mpl

import pandas as pd


import torch

from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset

from torchvision import transforms as T
from torchvision.datasets import MNIST, FashionMNIST

import torchmetrics
import pytorch_lightning as pl
from sklearn.metrics import ConfusionMatrixDisplay

In [3]:
#@title Variables

DEVICE="cpu"
NUM_CLASSES = 10
BATCH_SIZE = 64
NUM_WORKERS = 2
SEED = 1

MAIN_TRANSFORM = T.ToTensor()


ALL_CLASSES = list(range(10))
TASK_1_CLASSES = ALL_CLASSES[:5]
TASK_2_CLASSES = ALL_CLASSES[5:]

TRAINING_SAMPLES_BASIS = 6000

In [4]:
TASK_1_CLASSES, TASK_2_CLASSES

([0, 1, 2, 3, 4], [5, 6, 7, 8, 9])

# Construct Subset

In [5]:
def get_loaders(classes):
    data_class = FashionMNIST
    
    train_ds = data_class(
        root="./datasets", train=True, transform=MAIN_TRANSFORM, download=True
    )

    val_ds = data_class(
        root="./datasets", train=False, transform=MAIN_TRANSFORM, download=True
    )

    np.random.seed(1)

    
    arr_ds = []
    
    for ds in [train_ds, val_ds]:
        _ds = Subset(
            ds,
            indices=np.argwhere(np.isin(ds.targets, classes)).reshape(-1)
        )
        
        arr_ds.append(_ds)
        

    return DataLoader(arr_ds[0], num_workers=NUM_WORKERS, batch_size=BATCH_SIZE, shuffle=True), \
        DataLoader(arr_ds[1], num_workers=NUM_WORKERS, batch_size=BATCH_SIZE, shuffle=False), \

train_loader, val_loader = get_loaders(TASK_1_CLASSES)

In [23]:
class MLP(nn.Module):
    def __init__(self, input_channels=1):
        super().__init__()
        
        self.lin1 = nn.Linear(784, 128)
        self.act1 = nn.ReLU()
        
        self.lin2 = nn.Linear(128, 64)
        self.act2 = nn.ReLU()

        self.lin3 = nn.Linear(64, 32)
        self.act3 = nn.ReLU()
        
        self.lin4 = nn.Linear(32, 16)
        self.act4 = nn.ReLU()
        

        self.lin5 = nn.Linear(16, 10)
        
    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = self.act1(self.lin1(x))
        x = self.act2(self.lin2(x))
        x = self.act3(self.lin3(x))

        x = self.act4(self.lin4(x))

        x = self.lin5(x)

        return x


class ModelWrapper(pl.LightningModule):
    def __init__(self, model):
        super().__init__()

        self.model = model

        self.valid_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)

    def forward(self, x):
        embedding = self.model(x)
        return embedding

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optimizer

    def training_step(self, train_batch, batch_idx):
        x, y = train_batch

        yh = self.model(x)

        loss = F.cross_entropy(yh, y)
        
        if batch_idx % 200 == 0:
            print(loss)

        return loss
    
    def validation_step(self, val_batch, batch_idx):
        x, y = val_batch

        logits = self.model(x)
        self.valid_acc.update(logits, y)

    def on_validation_epoch_end(self):
        acc = self.valid_acc.compute()
        print(f"acc={acc}")
        self.log('valid_acc_epoch', self.valid_acc.compute())
        self.valid_acc.reset()


def attach_act_forward_hook(module):

    def fh(mod, input, output):

        assert isinstance(output, torch.Tensor)

        setattr(mod, "__output", output)
        output.retain_grad()

    hook = module.register_forward_hook(fh)

    return module, hook


def extract_features(model, layer,  dl, output_dir="./tmp", seed=1, verbose=False, is_saved=True):
    arr_act = []

    np.random.seed(seed)


    if verbose:

        print(f"Extracting activation from {layer}")


    try:
        module, hook = attach_act_forward_hook(getattr(model, layer))
        for bix, batch in enumerate(dl):
            x, y = batch

            logits = model(x)

            act = getattr(module, "__output")

            selected_act = act.detach().cpu().numpy()

            arr_act.append(selected_act)

    finally:
        hook.remove()

    arr_act = np.vstack(arr_act)


    if is_saved:
        output_dir = f"./output/{layer}"
        os.makedirs(output_dir, exist_ok=True)

        outer = arr_act.T @ arr_act / arr_act.shape[0]

        _, eigvecs = np.linalg.eigh(outer)

        eigvecs = eigvecs[:, ::-1]

        np.save(f"{output_dir}/eigvecs.npy", eigvecs)
        np.save(f"{output_dir}/eigvals.npy", eigvecs)


    return eigvecs
    
def attach_projected_fh_with_k(module, U):


    U = torch.from_numpy(U).float().to(DEVICE)


    def fh(mod, input, output):

        assert isinstance(output, torch.Tensor)

        A = U @ U.T
        I = torch.eye(A.shape[0])
#         a_on_U = output @ (I - A) + (output @ A).detach()
        a_on_U = output @ (I - A) 
#     + F.dropout((output@A), p=0.8).detach()

#         + (output @ A).detach()

#         np.testing.assert_allclose(a_on_U.detach().cpu(), output.detach().cpu(), atol=1e-3)

        return  a_on_U

    hook = module.register_forward_hook(fh)

    return hook

def train(model, epochs=5, projection=False):

    pl.seed_everything(SEED)

    start_time = datetime.now()

    
    arr_results = []
    
    for tix, task_classes in enumerate([TASK_1_CLASSES, TASK_2_CLASSES]):
        print(f"Training with Task {tix}: {task_classes} classes")

        train_loader, val_loader = get_loaders(task_classes)
        
        trainer = pl.Trainer(accelerator=DEVICE, max_epochs=epochs)

        arr_hooks = []
        try:
            if projection and tix > 0:
                k = 5
                print(f"Doing Projection with k={k}")
                
                for layer in ["act1", "act2", "act3", "act4"][3:]:
                    # extract_activation
                    _train_loader, _ = get_loaders(TASK_1_CLASSES)
                    U = extract_features(model, layer, _train_loader)

                    U = U[:, :k]

                    hook = attach_projected_fh_with_k(getattr(model, layer), U.copy())
                    arr_hooks.append(hook)

            trainer.fit(ModelWrapper(model), train_loader, val_loader)

        finally:
            for hook in arr_hooks:
                hook.remove()
        
        for evtix, _task_classes in enumerate([TASK_1_CLASSES, TASK_2_CLASSES]):
            valid_acc = torchmetrics.Accuracy(task="multiclass", num_classes=len(_task_classes))

            _, inner_val_loader =  get_loaders(_task_classes)

            for x, y in inner_val_loader:

                y = y - np.min(_task_classes)

                logits = model(x)[:, _task_classes]

                valid_acc.update(logits, y)

            acc = valid_acc.compute()

            print(f"Eval Task {evtix}: {_task_classes}: acc={acc:.4f}")


            arr_results.append(
                dict(
                    training_task=tix,
                    eval_task=evtix,
                    acc=float(acc)
                )
            )

    print("Time Took:", (datetime.now() - start_time) / 60, "mins")

    return pd.DataFrame(arr_results)


train(MLP(), epochs=5, projection=True)

Global seed set to 1
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs

  | Name      | Type               | Params
-------------------------------------------------
0 | model     | MLP                | 111 K 
1 | valid_acc | MulticlassAccuracy | 0     
-------------------------------------------------
111 K     Trainable params
0         Non-trainable params
111 K     Total params
0.446     Total estimated model params size (MB)


Training with Task 0: [0, 1, 2, 3, 4] classes


Sanity Checking: 0it [00:00, ?it/s]

/home/pat/.cache/pypoetry/virtualenvs/xaikd-bYmevfGI-py3.11/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:430: PossibleUserWarning: The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 8 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


acc=0.0


...

tensor(2.3398, grad_fn=<NllLossBackward0>)
tensor(0.4466, grad_fn=<NllLossBackward0>)
tensor(0.4105, grad_fn=<NllLossBackward0>)


Validation: 0it [00:00, ?it/s]

acc=0.8375999927520752
tensor(0.5770, grad_fn=<NllLossBackward0>)
tensor(0.2593, grad_fn=<NllLossBackward0>)
tensor(0.3168, grad_fn=<NllLossBackward0>)


Validation: 0it [00:00, ?it/s]

acc=0.8596000075340271
tensor(0.1994, grad_fn=<NllLossBackward0>)


In [22]:
F.dropout(torch.randn((20, 4)), p=0.9 )


tensor([[ 0.0000, -0.0000,  0.0000, -0.0000],
        [ 0.0000,  0.0000,  0.0000, -0.0000],
        [-0.0000,  0.0000, -0.0000, -0.0000],
        [ 0.0000,  0.0000, -0.0000, -0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000, -0.0000, 12.8578,  0.0000],
        [-0.0000, -0.0000, 11.2202,  0.0000],
        [ 0.0000, -2.3336, -0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000, -0.0000,  3.5386,  0.0000],
        [-7.1226, -0.0000, -0.0000, -0.0000],
        [-0.0000, -0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000, -0.0000,  0.0000],
        [-0.0000, -0.0000,  0.0000,  0.0000],
        [-0.0000, -0.0000,  0.0000, -0.0000],
        [ 5.1811,  0.0000,  0.0000,  0.0000],
        [ 0.3746, -6.3784, -0.0000, -0.0000],
        [ 0.0000, -0.0000, -0.0000, -0.0000],
        [ 0.0000, -0.0000, 17.9023,  0.0000],
        [ 0.0000,  0.0000,  0.0000, -0.0000]])

In [ ]:
train(MLP(), epochs=10, projection=False)

Global seed set to 1
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs

  | Name      | Type               | Params
-------------------------------------------------
0 | model     | MLP                | 111 K 
1 | valid_acc | MulticlassAccuracy | 0     
-------------------------------------------------
111 K     Trainable params
0         Non-trainable params
111 K     Total params
0.446     Total estimated model params size (MB)


Training with Task 0: [0, 1, 2, 3, 4] classes


Sanity Checking: 0it [00:00, ?it/s]

/home/pat/.cache/pypoetry/virtualenvs/xaikd-bYmevfGI-py3.11/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:430: PossibleUserWarning: The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 8 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


acc=0.2421875


...

Training: 0it [00:00, ?it/s]

tensor(2.2647, grad_fn=<NllLossBackward0>)
tensor(0.5636, grad_fn=<NllLossBackward0>)
tensor(0.4360, grad_fn=<NllLossBackward0>)


Validation: 0it [00:00, ?it/s]

acc=0.8334000110626221
tensor(0.6485, grad_fn=<NllLossBackward0>)
tensor(0.2376, grad_fn=<NllLossBackward0>)
tensor(0.3569, grad_fn=<NllLossBackward0>)


Validation: 0it [00:00, ?it/s]

In [ ]:
raise